# 03 - 数据处理对比: 手写 Dataset vs HF datasets

对比 from-scratch (手写 PyTorch Dataset) 与 HuggingFace (datasets 库 + DataCollator) 的数据处理方式。

| 维度 | from-scratch | HuggingFace |
|------|-------------|-------------|
| 数据加载 | 手写 JSON/JSONL 解析 | `datasets.load_dataset()` |
| Tokenize | `__getitem__` 中逐条处理 | `dataset.map(tokenize_fn, batched=True)` |
| Padding | 手动固定长度 padding | DataCollator 动态 padding |
| Labels | 手动构建 (shift / mask) | DataCollator 自动生成 |
| 与 Trainer | DataLoader 手动配置 | 直接传入 Trainer |

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import tempfile
from pathlib import Path

from data import ClearMindTokenizer, load_pretrain_dataset, load_sft_dataset, load_dpo_dataset

## 0. 准备测试数据和 Tokenizer

In [ ]:
# 准备 tokenizer
tokenizer_path = '../outputs/tokenizer'
if Path(tokenizer_path).exists():
    tokenizer = ClearMindTokenizer.load(tokenizer_path)
else:
    # 训练一个 tiny tokenizer 用于演示
    tmpdir = tempfile.mkdtemp()
    corpus = Path(tmpdir) / 'corpus.txt'
    corpus.write_text('\n'.join([
        '深度学习是机器学习的一个分支。', '自然语言处理是人工智能的重要领域。',
        'Transformer uses self-attention mechanisms.', '预训练语言模型学习数据表示。',
    ] * 10))
    tokenizer = ClearMindTokenizer.train(str(corpus), vocab_size=500)

print(f'Tokenizer vocab_size: {tokenizer.vocab_size}')
print(f'Special tokens: bos={tokenizer.bos_token}, eos={tokenizer.eos_token}, pad={tokenizer.pad_token}')

In [ ]:
# 创建临时测试数据
tmpdir = tempfile.mkdtemp()

# Pretrain 数据
pretrain_path = Path(tmpdir) / 'pretrain.jsonl'
with open(pretrain_path, 'w') as f:
    for text in ['深度学习使用多层神经网络。', '自然语言处理涵盖文本分类。',
                 'Transformer captures long-range dependencies.', '预训练模型学习语言理解。']:
        f.write(json.dumps({'text': text}, ensure_ascii=False) + '\n')

# SFT 数据
sft_path = Path(tmpdir) / 'sft.jsonl'
with open(sft_path, 'w') as f:
    for item in [
        {'instruction': '什么是深度学习？', 'input': '', 'output': '深度学习是机器学习的一个分支。'},
        {'instruction': '解释 Transformer', 'input': '', 'output': 'Transformer 使用自注意力机制。'},
    ]:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

# DPO 数据
dpo_path = Path(tmpdir) / 'dpo.jsonl'
with open(dpo_path, 'w') as f:
    for item in [
        {'prompt': '什么是 AI？', 'chosen': 'AI 是计算机科学的分支。', 'rejected': 'AI 就是电脑。'},
        {'prompt': '什么是 NLP？', 'chosen': 'NLP 是处理人类语言的技术。', 'rejected': 'NLP 是编程。'},
    ]:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f'测试数据已创建: {tmpdir}')

## 1. 预训练数据处理

In [ ]:
# === HuggingFace 方式 ===
# datasets.load_dataset → dataset.map(tokenize_fn) → DataCollator
pretrain_ds = load_pretrain_dataset(str(pretrain_path), tokenizer, max_length=64)

print(f'训练集: {pretrain_ds["train"]}')
print(f'验证集: {pretrain_ds["validation"]}')
print(f'\n样本 keys: {list(pretrain_ds["train"][0].keys())}')
print(f'input_ids 长度: {len(pretrain_ds["train"][0]["input_ids"])}')
print(f'input_ids[:10]: {pretrain_ds["train"][0]["input_ids"][:10]}')

# 对比: from-scratch 方式
# dataset = PretrainDataset(data_path, tokenizer, max_seq_len=64)
# 手动在 __getitem__ 中 tokenize + 拼接 + 切 chunk
# input_ids = chunk[:-1], labels = chunk[1:]  # 手动 shift

In [ ]:
# === DataCollator 自动生成 labels ===
from transformers import DataCollatorForLanguageModeling

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
batch = collator([pretrain_ds['train'][i] for i in range(min(2, len(pretrain_ds['train'])))])

print('DataCollator 输出:')
print(f'  input_ids shape: {batch["input_ids"].shape}')
print(f'  labels shape:    {batch["labels"].shape}')
print(f'  input_ids[0][:10]: {batch["input_ids"][0][:10].tolist()}')
print(f'  labels[0][:10]:    {batch["labels"][0][:10].tolist()}')

# labels 中 padding 位置为 -100
pad_id = tokenizer.pad_token_id
pad_mask = (batch['input_ids'] == pad_id)
print(f'\nPadding 位置 labels 全为 -100: {(batch["labels"][pad_mask] == -100).all().item()}')

# 对比: from-scratch 手动做 labels = input_ids[1:] (shift by 1)

## 2. SFT 数据处理

In [ ]:
# === HuggingFace 方式 ===
# apply_chat_template 格式化 → tokenize → labels mask
sft_ds = load_sft_dataset(str(sft_path), tokenizer, max_length=128)

print(f'训练集: {sft_ds["train"]}')
sample = sft_ds['train'][0]
print(f'\n样本 keys: {list(sample.keys())}')
print(f'input_ids 长度: {len(sample["input_ids"])}')

# 解码查看格式
decoded = tokenizer.decode(sample['input_ids'])
print(f'\n格式化文本:\n{decoded}')

In [ ]:
# === Labels Mask 可视化 ===
# prompt 部分: -100 (不计算 loss)
# response 部分: 有效 token id (计算 loss)
labels = sample['labels']
input_ids = sample['input_ids']

masked_count = sum(1 for l in labels if l == -100)
valid_count = sum(1 for l in labels if l != -100)
print(f'Labels 统计:')
print(f'  Masked (prompt, -100): {masked_count} tokens')
print(f'  Valid (response):      {valid_count} tokens')
print(f'  总长度:                {len(labels)} tokens')

# 找到 prompt/response 分界点
boundary = next((i for i, l in enumerate(labels) if l != -100), len(labels))
print(f'\nPrompt 部分 (0:{boundary}):')
print(f'  {tokenizer.decode(input_ids[:boundary])}')
print(f'Response 部分 ({boundary}:{len(labels)}):')
print(f'  {tokenizer.decode(input_ids[boundary:])}')

# 对比: from-scratch 手动计算 prompt_len
# prompt_ids = tokenizer.encode(prompt_text)
# labels[:len(prompt_ids)] = -100  # 手动 mask

In [ ]:
# === Chat Template 演示 ===
# HF tokenizer 内置 Jinja2 模板，自动格式化对话
messages = [
    {'role': 'user', 'content': '什么是深度学习？'},
    {'role': 'assistant', 'content': '深度学习是机器学习的一个分支。'},
]

# 不 tokenize，只看格式化结果
formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
print(f'完整对话格式:\n{repr(formatted)}')

# 只生成 prompt (用于推理)
prompt_only = tokenizer.apply_chat_template(
    [messages[0]], tokenize=False, add_generation_prompt=True
)
print(f'\nPrompt (with generation prompt):\n{repr(prompt_only)}')

# 对比: from-scratch 硬编码拼接
# prompt_text = f"Human: {instruction}\nAssistant: "
# full_text = f"Human: {instruction}\nAssistant: {response}"

## 3. DPO 数据处理

In [ ]:
# === HuggingFace 方式 ===
# 只需格式化为字符串，DPOTrainer 内部处理 tokenize
dpo_ds = load_dpo_dataset(str(dpo_path), tokenizer, max_length=128)

print(f'训练集: {dpo_ds["train"]}')
sample = dpo_ds['train'][0]
print(f'\n样本 keys: {list(sample.keys())}')
print(f'\nprompt:   {repr(sample["prompt"])}')
print(f'chosen:   {repr(sample["chosen"])}')
print(f'rejected: {repr(sample["rejected"])}')

# 对比: from-scratch 方式
# DPODataset 手动 tokenize chosen 和 rejected
# 返回 4 个 tensor: chosen_input_ids, chosen_labels, rejected_input_ids, rejected_labels
# 还需要手动创建 ref_model = copy.deepcopy(model)

In [ ]:
# DPO 数据是纯字符串格式 (不是 token ids)
# 这是 HuggingFace DPOTrainer 的标准输入
for sample in dpo_ds['train']:
    print(f'prompt 类型: {type(sample["prompt"]).__name__}, '
          f'chosen 类型: {type(sample["chosen"]).__name__}, '
          f'rejected 类型: {type(sample["rejected"]).__name__}')
    assert sample['chosen'] != sample['rejected']

print('\n所有样本都是字符串格式，chosen ≠ rejected ✓')

# 对比: from-scratch 返回的是 tokenized tensors
# {'chosen_input_ids': tensor, 'chosen_labels': tensor,
#  'rejected_input_ids': tensor, 'rejected_labels': tensor}

## 4. datasets 库的优势

In [ ]:
# === datasets 库特性演示 ===
from datasets import load_dataset

# 1. 懒加载 + Arrow 格式 — 内存映射，支持大规模数据
ds = load_dataset('json', data_files=str(pretrain_path), split='train')
print(f'Arrow 格式: {ds.format}')
print(f'特征: {ds.features}')
print(f'缓存文件: {ds.cache_files}')

# 2. batched map — 比逐条处理快很多
def tokenize_fn(examples):
    return tokenizer(examples['text'], truncation=True, max_length=64)

tokenized = ds.map(tokenize_fn, batched=True, remove_columns=['text'])
print(f'\nTokenized 特征: {tokenized.features}')
print(f'样本数: {len(tokenized)}')

# 3. train_test_split — 内置划分
splits = tokenized.train_test_split(test_size=0.5, seed=42)
print(f'\nTrain: {len(splits["train"])}, Test: {len(splits["test"])}')

# 对比: from-scratch
# 手写 JSON 解析、手动 shuffle + split、逐条 tokenize

## 5. 与 Trainer 集成

In [ ]:
# === HuggingFace Trainer 集成示例 (伪代码) ===

print('=== Pretrain (HF Trainer) ===')
print('''
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

dataset = load_pretrain_dataset(data_path, tokenizer, max_length=512)
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=TrainingArguments(...),
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=collator,        # 自动生成 labels + 动态 padding
)
trainer.train()
''')

print('=== SFT (TRL SFTTrainer) ===')
print('''
from trl import SFTTrainer, SFTConfig

dataset = load_sft_dataset(data_path, tokenizer, max_length=512)

trainer = SFTTrainer(
    model=model,
    args=SFTConfig(...),
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    # labels 已包含 -100 mask，直接用默认 collator
)
trainer.train()
''')

print('=== DPO (TRL DPOTrainer) ===')
print('''
from trl import DPOTrainer, DPOConfig

dataset = load_dpo_dataset(data_path, tokenizer, max_length=512)

trainer = DPOTrainer(
    model=model,
    args=DPOConfig(...),
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    # DPOTrainer 自动: tokenize, 创建 ref_model, 计算 DPO loss
)
trainer.train()
''')

# 对比: from-scratch 需要手写 training loop
# for batch in DataLoader(dataset):
#     logits = model(batch['input_ids'])
#     loss = F.cross_entropy(logits, batch['labels'])
#     loss.backward()
#     optimizer.step()

## 总结

| 功能 | from-scratch | HuggingFace |
|------|-------------|-------------|
| 数据加载 | 手写 JSON 解析 | `datasets.load_dataset()` |
| Tokenize | `__getitem__` 逐条处理 | `dataset.map(fn, batched=True)` 批量处理 |
| Padding | 固定长度, 手动填充 | DataCollator 动态 padding |
| Labels (Pretrain) | `input_ids[1:]` 手动 shift | `DataCollatorForLanguageModeling` 自动 |
| Labels (SFT) | 手动计算 prompt_len | `apply_chat_template` + mask |
| DPO 数据 | 手动 tokenize 4 组 tensor | 字符串格式, DPOTrainer 内部处理 |
| 内存管理 | 全部加载到内存 | Arrow 格式, 内存映射 |
| Train/Val 划分 | 手动 shuffle + split | `train_test_split()` |
| Trainer 集成 | 手写 training loop | 直接传入 Trainer |

**核心收获:** 数据处理的核心逻辑（tokenize、labels mask、padding）完全一致，
HuggingFace 通过 datasets 库 + DataCollator 将这些操作标准化和自动化，
减少了手写代码量，同时支持更大规模的数据集（Arrow 内存映射）。